## Parallel Workflow with Microsoft Agent Framework and Microsoft Foundry

![parallel-workflow](./Assets/parallel_workflow.png)

In [2]:
%pip install python-dotenv azure-identity agent-framework==1.0.0rc2 opentelemetry-semantic-conventions-ai==0.4.13 azure-ai-projects==2.0.0b3 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\vijay\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Setting Up the Environment

In [4]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from agent_framework import Executor, WorkflowBuilder, WorkflowContext, handler
from agent_framework import WorkflowBuilder, WorkflowViz

load_dotenv()
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model = os.getenv("AI_FOUNDRY_DEPLOYMENT_NAME")

print("Project Endpoint: ", project_endpoint)
print("Model: ", model)

Project Endpoint:  https://tavant-mf.services.ai.azure.com/api/projects/proj-default
Model:  gpt-4o


### Defining a Function to Create a Chat Agent

In [6]:
from agent_framework import Agent
from agent_framework.azure import AzureAIClient
from azure.ai.projects.aio import AIProjectClient
from azure.identity.aio import AzureCliCredential

# Cell 2: Define async workflow
async def create_agent(agent_name: str,
                       agent_instructions: str) -> Agent:
    
    # Create async Azure credential
    credential = AzureCliCredential()

    # creating the Foundry Project Client
    project_client = AIProjectClient(
        endpoint=project_endpoint,
        credential=credential
    )

    # creating a conversation using the OpenAI Client
    openai_client = project_client.get_openai_client()
    conversation = await openai_client.conversations.create()
    conversation_id = conversation.id
    print("Conversation ID: ", conversation_id)
    
    # Initialize the Azure AI Agent Client
    chat_client = AzureAIClient(project_client=project_client,
                                conversation_id=conversation_id,
                                model_deployment_name=model)

    try:
        agent = chat_client.as_agent(
            name=agent_name,
            instructions=agent_instructions,
        )

        print("{} Agent created successfully!".format(agent_name))
        return agent

    finally:
        # Clean up async clients
        await chat_client.close()
        await credential.close()

### Creating the Location Picker Agent

In [7]:
location_picker_agent = await create_agent(
    agent_name = "Location-Picker-Agent",
    agent_instructions = """You are a helpful assistant that helps users pick a location for their vacation."""
)

Conversation ID:  conv_2dc501033dcea1a200sHZ55nUF7q6sXgAZO3iqeqvNBTI9g5ot
Location-Picker-Agent Agent created successfully!


### Creating the Destination Recommender Agent

In [8]:
destination_recommender_agent = await create_agent(
    agent_name = "Destination-Recommender-Agent",
    agent_instructions = """You are a travel expert that provides personalized vacation recommendations based on user preferences and locations. """
)

Conversation ID:  conv_a3bb86d33dadea1d00kEzVeOdE8DrM7eJ3hTRkTeqXNfdEqZJO
Destination-Recommender-Agent Agent created successfully!


### Creating the Weather Agent

In [9]:
weather_agent = await create_agent(
    agent_name = "Weather-Agent",
    agent_instructions = """You are a weather expert that provides accurate and up-to-date weather information for various locations selected"""
)

Conversation ID:  conv_cfbb6cb7e6dff7cf007qAbV4OciRyT8tgDk10ra8GwFnoJdCkG
Weather-Agent Agent created successfully!


### Creating the Cuisine Suggestion Agent

In [10]:
cuisine_suggestion_agent = await create_agent(
    agent_name = "Cuisine-Suggestion-Agent",
    agent_instructions = """You are a culinary expert that suggests popular local cuisines and dining options based on the selected vacation destinations."""
)

Conversation ID:  conv_d59446728778072e00K5UB8Ey0HlwKMTNlgIzwWWwNVKZkiq71
Cuisine-Suggestion-Agent Agent created successfully!


### Creating the Itinerary Planner Agent

In [11]:
itinerary_planner_agent = await create_agent(
    agent_name = "Itinerary-Planner-Agent",
    agent_instructions = """You are an itinerary planning expert that creates detailed travel itineraries based on user preferences, selected destinations, weather conditions, and local cuisine options."""
)

Conversation ID:  conv_7293a95cd62ad1ef00yr0jThlDcYd9dhlJwqrvWHotm1z1pUFW
Itinerary-Planner-Agent Agent created successfully!


### Creating the Location Selector Agent Executor

In [12]:
class LocationSelectorExecutor(Executor):

    @handler
    async def handle(self, user_query: str, ctx: WorkflowContext[str]) -> None:
        response = await location_picker_agent.run(user_query)

        await ctx.send_message(str(response))

### Creating the Destination Recommender Agent Executor

In [13]:
class DestinationRecommenderExecutor(Executor):

    @handler
    async def handle(self, location: str, ctx: WorkflowContext[str]) -> None:
        response = await destination_recommender_agent.run(location)

        await ctx.send_message(str(response))

### Creating the Weather Agent Executor

In [14]:
class WeatherExecutor(Executor):

    @handler
    async def handle(self, location: str, ctx: WorkflowContext[str]) -> None:
        response = await weather_agent.run(location)

        await ctx.send_message(str(response))

### Creating the Cuisine Suggestion Agent Executor

In [15]:
class CuisineSuggestionExecutor(Executor):

    @handler
    async def handle(self, location: str, ctx: WorkflowContext[str]) -> None:
        response = await cuisine_suggestion_agent.run(location)

        await ctx.send_message(str(response))

### Creating the Itinerary Planner Agent Executor

In [30]:
class ItineraryPlannerExecutor(Executor):

    @handler
    async def handle(self, results: list[str], ctx: WorkflowContext[str]) -> None:
        response = await itinerary_planner_agent.run(results)

        await ctx.yield_output(str(response))

### Building the Parallel Workflow

In [ ]:
# Create the executor instances / objects
location_selector_executor = LocationSelectorExecutor(id="LocationSelector")
destination_recommender_executor = DestinationRecommenderExecutor(id="DestinationRecommender")
weather_executor = WeatherExecutor(id="Weather")
cuisine_suggestion_executor = CuisineSuggestionExecutor(id="CuisineSuggestion")
itinerary_planner_executor = ItineraryPlannerExecutor(id="ItineraryPlanner")

# Build the workflow
workflow = (
    WorkflowBuilder(start_executor=location_selector_executor)
    .add_fan_out_edges(location_selector_executor, [destination_recommender_executor, weather_executor, cuisine_suggestion_executor])
    .add_fan_in_edges([destination_recommender_executor, weather_executor, cuisine_suggestion_executor], itinerary_planner_executor)
    .build()
)

viz = WorkflowViz(workflow)

### Generating Mermaid Diagram for Visualization

In [32]:
mermaid_content = viz.to_mermaid()

# printing mermaid content as markdown
from IPython.display import Markdown, display
display(Markdown(f"```mermaid\n{mermaid_content}\n```"))

```mermaid
flowchart TD
  LocationSelector["LocationSelector (Start)"];
  DestinationRecommender["DestinationRecommender"];
  Weather["Weather"];
  CuisineSuggestion["CuisineSuggestion"];
  ItineraryPlanner["ItineraryPlanner"];
  LocationSelector --> DestinationRecommender;
  LocationSelector --> Weather;
  LocationSelector --> CuisineSuggestion;
  DestinationRecommender --> ItineraryPlanner;
  Weather --> ItineraryPlanner;
  CuisineSuggestion --> ItineraryPlanner;
```

### Running the Workflow and Streaming Events

In [33]:
# Run the workflow and stream events in notebook
async def main():
    result = workflow.run(
        "help me plan a vacation to India with the following details: I love historical sites",
        stream=True,
    )

    async for event in result:
        print(f"Event: {event}")

    final = await result.get_final_response()
    outputs = final.get_outputs() if hasattr(final, "get_outputs") else []
    if outputs:
        print(f"Workflow completed with result: {outputs[-1]}")

await main()

Event: WorkflowEvent(type='started')
Event: WorkflowEvent(type='status', state=IN_PROGRESS)
Event: WorkflowEvent(type='executor_invoked', executor_id='LocationSelector', data='help me plan a vacation to India with the following details: I love historical sites')
Event: WorkflowEvent(type='executor_completed', executor_id='LocationSelector', data=['Certainly! Here’s a detailed itinerary focusing solely on historical sites across India:\n\n### Destinations:\n\n1. **Delhi**\n   - **Red Fort**: Explore the massive fortifications and learn about Mughal history.\n   - **Humayun’s Tomb**: Admire the architectural beauty and serene gardens.\n   - **Qutub Minar**: Discover the history behind this ancient minaret.\n\n2. **Agra**\n   - **Taj Mahal**: Visit this iconic symbol of love and a UNESCO World Heritage site.\n   - **Agra Fort**: Explore the royal chambers and gardens.\n   - **Fatehpur Sikri**: Wander through this ancient city with stunning palaces and mosques.\n\n3. **Jaipur**\n   - **Amb